# Mining

Transactions transfer bitcoins from one party to another and are unlocked, or authorized by signatures, ensuring that the sender authorized the transaction.

In traditional physical cash transactions, once you hand over a bill to someone, you no longer possess that bill. However, in the digital realm, the "money" is essentially just information that can be easily copied. This creates a potential problem where a person could copy their digital money and spend it multiple times.

Bitcoin, was the first system to effectively solve this problem. It uses blocks to order transactions, if we order transactions, a double-spend can be prevented by making any later conflicting transaction invalid, accepting only the earlier transaction as the valid one.

Implementing this approach can cause trasmission overhead in coming to consesus, because the different nodes of the network must agree on which transaction is supposed to be next.
The solution to this problem was found by settling every 10 minutes in batches of transactions. These batches of transactions are called blocks.

Coinbase transactions are required as first transaction of every block and is the only transaction allowed to bring bitcoins into existence.

The coinbase transaction's outputs are kept by whomever the mining entity designates and usually include all the transaction fees of the order transactions in the block and it's called the *block reward*.


In [1]:
from helper import (
    bits_to_target,
    hash256,
    int_to_little_endian,
    little_endian_to_int,
    merkle_root
)

In [2]:
# Constants
GENESIS_BLOCK = bytes.fromhex('0100000000000000000000000000000000000000000000000000000000000000000000003ba3edfd7a7b12b27ac72c3e67768f617fc81bc3888a51323a9fb8aa4b1e5e4a29ab5f49ffff001d1dac2b7c')
TESTNET_GENESIS_BLOCK = bytes.fromhex('0100000000000000000000000000000000000000000000000000000000000000000000003ba3edfd7a7b12b27ac72c3e67768f617fc81bc3888a51323a9fb8aa4b1e5e4adae5494dffff001d1aa4ae18')
LOWEST_BITS = bytes.fromhex('ffff001d')

In [3]:
class Block:

    def __init__(self, version, prev_block, merkle_root, timestamp, bits, nonce, tx_hashes=None):
        self.version = version
        self.prev_block = prev_block
        self.merkle_root = merkle_root
        self.timestamp = timestamp
        self.bits = bits
        self.nonce = nonce
        self.tx_hashes = tx_hashes
    
    @classmethod
    def parse(cls, s):
        ''' Takes a byte stream and parses a block. Returns a Block object '''
        # version - 4 bytes, little indian intrepreted as int
        version = little_endian_to_int(s.read(4))
        # prev_block - 32 bytes little indian, [::-1] used to reverse
        prev_block = s.read(32)[::-1]
        # merkle_root - 32 bytes little indian, [::-1] used to reverse
        merkle_root = s.read(32)[::-1]
        # timestamp - 4 bytes, little indian as int
        timestamp = little_endian_to_int(s.read(4))
        # bits - 4 bytes
        bits = s.read(4)
        # nonce - 4 bytes
        nonce = s.read(4)
        # initialize
        return cls(version, prev_block, merkle_root, timestamp, bits, nonce)

    
    def serialize(self):
        ''' returns the 80 byte block header '''
        result = int_to_little_endian(self.version, 4)
        result += self.prev_block[::-1]
        result += self.merkle_root[::-1]
        result += int_to_little_endian(self.timestamp, 4)
        result += self.bits
        result += self.nonce
        return result
    
    def hash(self):
        ''' Returns hashed little endian interpreted of the block '''
        s = self.serialize()
        h256 = hash256(s)
        return h256[::-1]
    
    def bip9(self):
        ''' Returns whether this block is signaling readiness for BIP9 '''
        return self.version >> 29 == 0b001
    
    def bip91(self):
        ''' Returns whether this block is signaling readiness for BIP91 '''
        return self.version >> 4 & 1 == 1
    
    def bip141(self):
        ''' Returns whether this block is signaling readiness for BIP141 '''
        return self.version >> 1 & 1 == 1
    
    def target(self):
        ''' Returns the proof-of-work target based on bits '''
        return bits_to_target(self.bits)
    
    def check_pow(self):
        ''' Verifies that the block satisfies the proof-of-work requirement '''
        h = self.hash()
        proof = little_endian_to_int(h)
        return proof < self.target()
    
    def validate_merkle_root(self):
        ''' Validates the merkle root of the block '''
        hashes = [h[::-1] for h in self.tx_hashes]
        root = merkle_root(hashes)[::-1]
        return root == self.merkle_root

## Proof-of-Work

*Proof-of-Work* is what secures Bitcoin and allows decentralized mining of Bitcoin. Find a *proof-of-work* gives a miner the right to put the attached block into the blockchain.
*Proof-of-Work* is called *"mining"* because like the physical mining, there is something that miners are searching for. *Proof-of-Work* is a number that provides a very rare result, similarly to gold mining that requires to process a lot of dirt and rocks before accumulating a small amount of gold. Once gold is found, it's very easy to verify that the gold is real. Like with gold, verifying proof-of-work is much cheaper than actually finding it.

Sha256 is known to generate uniformly distributed values. The probability of the first bit in a 256-bit number being 0 is 0.5, the first two bits being 00, 0.25, the first three bits being 000, 0.125, and so on.
The process of finding the proof-of-work requires us to process around 10^22 numerical bits to find our numerical gold nugget.

The nonce is a spare field at the end of the block header, the miners can change the nonce field at will to change the hash of the block header.
*Proof-of-Work* is the requirement that the hash of every block header in Bitcoin must be below a certain target. The target is a 256 bit number that is computed directly from the bits field. The target is very small compared to an average 256 bit number.
The bits field is actually two different numbers, the first is the exponend, which is the last byte and, the second is the coefficient, which is the other three bytes in little-endian.
The formula that describes the target is:
```
target = coefficient x 256^(exponent - 3)
```

In [9]:
# Calculate the target given the bits field

from helper import little_endian_to_int

bits = bytes.fromhex('e93c0118')
exponent = bits[-1]
coefficient = little_endian_to_int(bits[:-1])
target = coefficient * 256**(exponent - 3)

print(f'{target:064x}')

0000000000000000013ce9000000000000000000000000000000000000000000


A valid proof-of-work is a hash of the block header that, when interpreted as a little-endian integer, is below the target number. Proof-of-work hashes are exceedingly rare, and the process of mining is the process of finding one of these hashes.


In [10]:
proof = little_endian_to_int(hash256(bytes.fromhex('020000208ec39428b17323\
fa0ddec8e887b4a7c53b8c0a0a220cfd0000000000000000005b0750fce0a889502d40508d3957\
6821155e9c9e3f5c3157f961db38fd8b25be1e77a759e93c0118a4ffd71d')))

print(proof < target)

True


Targets are hard for human beings to comprehend, because it's not easy to see the difference between a 180-bit number and a 190-bit number.
To make different targets easier to compare, the concept of difficulty was born. The difficulty is inversely proportional to the target and the formula is:

```
difficulty = 0xffff x 256^(0x1d - 3) / target
```

In [11]:
bits = bytes.fromhex('e93c0118')

exponent = bits[-1]

coefficient = little_endian_to_int(bits[:-1])

target = coefficient*256**(exponent-3)

difficulty = 0xffff * 256**(0x1d-3) / target

print(difficulty)

888171856257.3206


In Bitcoin, each group of 2016 blocks is called *difficulty adjustment period*. At the end of every *difficulty adjustment period*, the target is adjusted according to this formula:

```
time_differential = (block timestamp of last block in difficulty adjustment period) - (block timestamp of first block in difficulty adjustment period)

new_target = previous_target * time_differential / (2 weeks)
```

The *time_differential* is calculated so that if it's greater than 8 weeks, 8 weeks is used, and if it's less than 3.5 days, 3.5 days is used. If each block took on average 10 minutes to create, 2016 blocks should take 20160 minutes. There are 1440 minutes per day, which means that 2016 blocks will take 14 days to create.